# RiverGuard v2: training on Kaggle

Trains the plastic-waste detector on the merged dataset built by `training/build_dataset.py`
(attached to this notebook as a Kaggle dataset), then scores it on the held-out test split.
Everything in `/kaggle/working/output` is what `kaggle kernels output` downloads.

In [ ]:
# Settings
RUN = "v4"
# Start from init.pt if the dataset has one (v4: v1's weights, so it keeps what v1 knows and only learns the
# additions), else from COCO-pretrained YOLO11s. Same size as v1, so the free CPU demo stays fast.
BASE_WEIGHTS = "yolo11s.pt"
EPOCHS = 80
PATIENCE = 20                 # stop early if val mAP hasn't improved for this many epochs
IMGSZ = 640
BATCH = 32
ULTRALYTICS_VERSION = "8.4.163"  # match the app's version for reproducibility

In [ ]:
%pip install -q "ultralytics=={ULTRALYTICS_VERSION}"

In [ ]:
# Find the attached dataset (Kaggle may mount it extracted or as a zip) and copy it somewhere writable.
import shutil, zipfile
from pathlib import Path

DATA = Path("/kaggle/working/data")
if not (DATA / "data.yaml").exists():
    yamls = list(Path("/kaggle/input").rglob("data.yaml"))
    if yamls:
        shutil.copytree(yamls[0].parent, DATA, dirs_exist_ok=True)
    else:
        archive = next(Path("/kaggle/input").rglob("*.zip"))
        zipfile.ZipFile(archive).extractall(DATA)
print((DATA / "data.yaml").read_text())
print({s: len(list((DATA / s / "images").glob("*"))) for s in ("train", "val", "test")})

In [ ]:
import torch
from ultralytics import YOLO

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable GPU in settings!)")
init = next(Path("/kaggle/input").rglob("init.pt"), None)
print("Starting from", init or BASE_WEIGHTS)
model = YOLO(str(init) if init else BASE_WEIGHTS)
model.train(
    data=str(DATA / "data.yaml"), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=4, seed=0, cos_lr=True,
    project="/kaggle/working/runs", name=RUN, exist_ok=True,
)

In [ ]:
# Score the best checkpoint on the held-out test split (never seen during training).
import json
best = YOLO(f"/kaggle/working/runs/{RUN}/weights/best.pt")
m = best.val(data=str(DATA / "data.yaml"), split="test", imgsz=IMGSZ, plots=True,
             project="/kaggle/working/runs", name=f"{RUN}-test", exist_ok=True)
test_metrics = {"mAP50": m.box.map50, "mAP50-95": m.box.map, "precision": m.box.mp, "recall": m.box.mr,
                "per_class_mAP50-95": dict(zip(best.names.values(), m.box.maps.tolist()))}
print(json.dumps(test_metrics, indent=2))

In [ ]:
out = Path("/kaggle/working/output")
out.mkdir(exist_ok=True)
run = Path(f"/kaggle/working/runs/{RUN}")
shutil.copy(run / "weights" / "best.pt", out / "best.pt")
for f in ["results.csv", "results.png", "confusion_matrix.png", "args.yaml"]:
    if (run / f).exists():
        shutil.copy(run / f, out / f)
(out / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
shutil.rmtree("/kaggle/working/data")  # keep the downloadable output small
print(sorted(p.name for p in out.iterdir()))